In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
loan = spark.table("pysparkdbt.silver.loan_data")
repayment = spark.table("pysparkdbt.silver.repaymemt_data")
sch_loan = spark.table("pysparkdbt.silver.sch_loan")

In [0]:
cur_rep = repayment.filter(
    (trunc(col("due_date"), "month") == trunc(current_date(), "month")) &
    (trunc(col("payment_date"), "month") == trunc(current_date(), "month"))
)


### Curent Month Demand & Collection

In [0]:
month_repayment = (
    loan.select(
        "customer_id", "loan_id", "loan_status", "loan_amount"
    )
    .join(
       cur_rep.select("loan_id", "due_date","amount_due","payment_date", "amount_paid","payment_status"),
        "loan_id",
        "left"
    )
)

### loan wise total due & Collection

In [0]:
collection=repayment.filter(col("payment_status")=="Paid").groupBy("loan_id").agg(sum("amount_due").alias("total_due"),sum("amount_paid").alias("total_paid"))

In [0]:
collection.write\
          .format("delta")\
          .mode("overwrite")\
          .saveAsTable("pysparkdbt.gold.collection_summary")

month_repayment.write\
          .format("delta")\
          .mode("overwrite")\
          .saveAsTable("pysparkdbt.gold.month_repayment")